In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv
/kaggle/input/datasets/mrinalpandey2/mcq-finall/adapter_model.safetensors
/kaggle/input/datasets/mrinalpandey2/mcq-finall/adapter_config.json


In [3]:
!pip install -Uq "trl[peft]" bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 47.3 MB/s eta 0:00:00


In [4]:
# ════════════════════════════════════════════════════════════════════════════
# ALL fast setup — imports, paths, data, constants, all prompt builders.
# Re-run only this cell + cell 3 after any kernel restart.
# ════════════════════════════════════════════════════════════════════════════

import os, gc
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
LORA_PATH     = "/kaggle/input/datasets/mrinalpandey2/mcq-finall"
COMP_PATH     = "/kaggle/input/competitions/smart-mcq-solver-challenge"

train = pd.read_csv(f"{COMP_PATH}/train.csv")
test  = pd.read_csv(f"{COMP_PATH}/test.csv")
print(f"train: {train.shape}  |  test: {test.shape}")
print("Answer distribution:\n", train['answer'].value_counts())

OPTION_LETTERS = ["A", "B", "C", "D", "E"]

PREFIXES = [
    "Pick the best possible answer:",
    "Determine the correct option:",
    "Select the most accurate option:",
    "Identify the correct statement:",
]

# 5 circular perms: each original option sits in each display position
# exactly once → positional bias cancels when logits are summed.
FIXED_PERMS = [
    ["A","B","C","D","E"],
    ["B","C","D","E","A"],
    ["C","D","E","A","B"],
    ["D","E","A","B","C"],
    ["E","A","B","C","D"],
]

def clean_prompt(prompt):
    prompt = prompt.strip()
    for p in PREFIXES:
        if prompt.lower().startswith(p.lower()):
            return prompt[len(p):].strip()
    return prompt

# ── Method 1: PLM prompt (permuted options, NO few-shot) ─────────────────────
# Few-shot was removed because it showed options in original A-E order while
# the main question uses permuted order — contradicting itself in 4/5 perms.
def build_plm_prompt(row, perm):
    q    = clean_prompt(row['prompt'])
    opts = '\n'.join(f"{pos}. {row[orig]}"
                     for pos, orig in zip(OPTION_LETTERS, perm))
    return (
        "You are an expert at solving multiple-choice questions.\n\n"
        "Choose the single best answer.\n\n"
        f"Question:\n{q}\n\n"
        f"Options:\n{opts}\n\n"
        "Answer with exactly one uppercase letter.\n\nAnswer:"
    )

# ── Method 2: CoT prompt (reasoning chain, original option order) ─────────────
# Ask the model to reason before committing. We generate ~60 tokens of
# reasoning, then append a suffix and read logits at that position.
# Using original option order here (no permutation) — the reasoning chain
# itself is what grounds the prediction, not position debiasing.
def build_cot_prompt(row):
    q    = clean_prompt(row['prompt'])
    opts = '\n'.join(f"{l}. {row[l]}" for l in OPTION_LETTERS)
    return (
        "You are an expert. Carefully analyze the question and each option.\n\n"
        f"Question:\n{q}\n\n"
        f"Options:\n{opts}\n\n"
        "Think step by step, then state which option is correct."
    )

COT_SUFFIX = "\n\nTherefore, the best answer is:"

# ── Method 3: Verification prompt (one per option) ───────────────────────────
# Ask 'Is [option text] correct? Yes or No.' for each option independently.
# P(Yes) is the score. Zero label/position bias — evaluates content directly.
def build_verify_prompt(row, letter):
    q = clean_prompt(row['prompt'])
    return (
        f"Question: {q}\n\n"
        f"Proposed answer: {row[letter]}\n\n"
        "Is the proposed answer correct for the question above? "
        "Respond with Yes or No."
    )

# ── MAP@3 ─────────────────────────────────────────────────────────────────────
def map_at_3(preds, labels):
    scores = []
    for pred, label in zip(preds, labels):
        p = pred.strip().split()[:3]
        hit = [1.0/(i+1) for i,l in enumerate(p) if l == label]
        scores.append(hit[0] if hit else 0.0)
    return float(np.mean(scores))

print("Setup complete.")


train: (2000, 8)  |  test: (500, 7)
Answer distribution:
 answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Setup complete.


In [5]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
print("Tokenizer loaded")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading base model …")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
)

print("Attaching LoRA …")
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()
model.config.use_cache = True

dev = next(model.parameters()).device
for i in range(torch.cuda.device_count()):
    g = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {g.name}  {g.total_memory/1e9:.0f} GB  "
          f"reserved={torch.cuda.memory_reserved(i)/1e9:.1f} GB")

# Letter token ids
def letter_token_id(letter):
    for candidate in (letter, " " + letter):
        ids = tokenizer.encode(candidate, add_special_tokens=False)
        if len(ids) == 1:
            return ids[0]
    return tokenizer.encode(letter, add_special_tokens=False)[-1]

option_token_ids = {l: letter_token_id(l) for l in OPTION_LETTERS}
print("Option token ids:", option_token_ids)

# Yes/No token ids for Verify
yes_token_id = tokenizer.encode("Yes", add_special_tokens=False)[-1]
no_token_id  = tokenizer.encode("No",  add_special_tokens=False)[-1]
print(f"Yes token: {yes_token_id}  No token: {no_token_id}")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Attaching LoRA …
GPU 0: Tesla T4  16 GB  reserved=2.0 GB
GPU 1: Tesla T4  16 GB  reserved=7.0 GB
Option token ids: {'A': 32, 'B': 33, 'C': 34, 'D': 35, 'E': 36}
Yes token: 9454  No token: 2753


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# METHOD 1 — PLM (Permuted Letter-logit)
# Accumulates raw logits across 5 circular permutations.
# Temperature applied analytically afterwards via raw_to_preds().
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def score_plm(rows, batch_size=8, perms=FIXED_PERMS):
    all_raw = [{l: 0.0 for l in OPTION_LETTERS} for _ in rows]
    opt_ids = torch.tensor([option_token_ids[l] for l in OPTION_LETTERS], device=dev)

    for perm in tqdm(perms, desc="PLM perms", leave=True):
        for start in range(0, len(rows), batch_size):
            batch = rows[start:start+batch_size]
            idx   = list(range(start, min(start+batch_size, len(rows))))
            texts = [
                tokenizer.apply_chat_template(
                    [{"role":"user","content":build_plm_prompt(row,perm)}],
                    tokenize=False, add_generation_prompt=True,
                ) for row in batch
            ]
            enc = tokenizer(texts, return_tensors="pt", padding=True,
                            truncation=True, max_length=768).to(dev)
            logits     = model(**enc).logits[:, -1, :]      # (B, vocab)
            opt_logits = logits[:, opt_ids]                  # (B, 5)
            for bi, gi in enumerate(idx):
                for pi, pos in enumerate(OPTION_LETTERS):
                    all_raw[gi][perm[pi]] += opt_logits[bi, pi].item()
    return all_raw


# ══════════════════════════════════════════════════════════════════════════════
# METHOD 2 — CoT (Chain-of-Thought + logit)
# Generates 60-token reasoning chain, then reads letter logit at
# 'Therefore, the best answer is:' — model thinks before deciding.
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def score_cot(rows, batch_size=4, max_new_tokens=60):
    all_scores = []
    opt_ids    = torch.tensor([option_token_ids[l] for l in OPTION_LETTERS], device=dev)
    suffix_ids = tokenizer.encode(COT_SUFFIX, add_special_tokens=False)
    suffix_t   = torch.tensor(suffix_ids, device=dev)

    for start in tqdm(range(0, len(rows), batch_size), desc="CoT"):
        batch = rows[start:start+batch_size]

        texts = [
            tokenizer.apply_chat_template(
                [{"role":"user","content":build_cot_prompt(row)}],
                tokenize=False, add_generation_prompt=True,
            ) for row in batch
        ]
        enc = tokenizer(texts, return_tensors="pt", padding=True,
                        truncation=True, max_length=640).to(dev)

        # Step 1: generate reasoning chain (greedy, up to 60 tokens)
        gen_ids = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )  # (B, prompt_len + gen_len)

        # Step 2: append suffix to all generated sequences, batch forward pass
        suffix_exp = suffix_t.unsqueeze(0).expand(gen_ids.size(0), -1)  # (B, suf)
        full_ids   = torch.cat([gen_ids, suffix_exp], dim=1)             # (B, total)

        out       = model(input_ids=full_ids)
        last_lgt  = out.logits[:, -1, :]        # (B, vocab)
        opt_lgt   = last_lgt[:, opt_ids]        # (B, 5)

        for bi in range(gen_ids.size(0)):
            all_scores.append(
                {l: opt_lgt[bi, pi].item() for pi,l in enumerate(OPTION_LETTERS)}
            )
    return all_scores


# ══════════════════════════════════════════════════════════════════════════════
# METHOD 3 — Verify (binary Yes/No per option)
# For each option X: score = log P(Yes | 'Is X correct?')
# Completely independent of option order — evaluates content directly.
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def score_verify(rows):
    all_scores = []

    for row in tqdm(rows, desc="Verify"):
        # 5 prompts (one per option) as a single mini-batch
        texts = [
            tokenizer.apply_chat_template(
                [{"role":"user","content":build_verify_prompt(row, l)}],
                tokenize=False, add_generation_prompt=True,
            ) for l in OPTION_LETTERS
        ]
        enc = tokenizer(texts, return_tensors="pt", padding=True,
                        truncation=True, max_length=512).to(dev)

        logits    = model(**enc).logits[:, -1, :]    # (5, vocab)
        yes_lgt   = logits[:, yes_token_id]           # (5,)
        no_lgt    = logits[:, no_token_id]            # (5,)

        # log P(Yes | {Yes, No}) — numerically stable
        log_yes = yes_lgt - torch.log(
            torch.exp(yes_lgt) + torch.exp(no_lgt)
        )  # (5,)

        all_scores.append(
            {l: log_yes[i].item() for i,l in enumerate(OPTION_LETTERS)}
        )
    return all_scores


# ── Normalise + combine scores ────────────────────────────────────────────────
def norm(scores_dict):
    """Z-score normalise a {A:float...} dict so all methods are on same scale."""
    arr  = np.array([scores_dict[l] for l in OPTION_LETTERS], dtype=np.float64)
    std  = arr.std() + 1e-8
    return {l: (scores_dict[l] - arr.mean()) / std for l in OPTION_LETTERS}

def ensemble(plm_raw, cot_raw, ver_raw, plm_temp=1.0, w_plm=0.4, w_cot=0.4, w_ver=0.2):
    """
    Combine the three scoring methods into top-3 predictions.
    plm_raw : list of {A:float} accumulated raw logits (summed across perms)
    cot_raw : list of {A:float} single-pass logits after reasoning chain
    ver_raw : list of {A:float} log P(Yes) per option
    """
    preds = []
    for ps, cs, vs in zip(plm_raw, cot_raw, ver_raw):
        # Apply temperature to PLM (raw logit sum → softmax)
        arr_p = np.array([ps[l] for l in OPTION_LETTERS]) / plm_temp
        arr_p -= arr_p.max()
        prob_p = np.exp(arr_p); prob_p /= prob_p.sum()

        # Normalise CoT and Verify to z-scores, then softmax
        n_c = norm(cs); arr_c = np.array([n_c[l] for l in OPTION_LETTERS])
        arr_c -= arr_c.max(); prob_c = np.exp(arr_c); prob_c /= prob_c.sum()

        n_v = norm(vs); arr_v = np.array([n_v[l] for l in OPTION_LETTERS])
        arr_v -= arr_v.max(); prob_v = np.exp(arr_v); prob_v /= prob_v.sum()

        combined = w_plm*prob_p + w_cot*prob_c + w_ver*prob_v
        ranked   = [OPTION_LETTERS[i] for i in np.argsort(combined)[::-1]]
        preds.append(" ".join(ranked[:3]))
    return preds


In [7]:
# ── Run all 3 methods on 200 training rows (ONE pass each) ───────────────────
CALIB_N    = 200
CALIB_SEED = 99

calib_df     = train.sample(CALIB_N, random_state=CALIB_SEED).reset_index(drop=True)
calib_rows   = [row for _,row in calib_df.iterrows()]
calib_labels = calib_df['answer'].tolist()

print(f"Running 3 scoring methods on {CALIB_N} calib rows …")

print("[1/3] PLM …")
calib_plm = score_plm(calib_rows, batch_size=8)

print("[2/3] CoT …")
calib_cot = score_cot(calib_rows, batch_size=4)

print("[3/3] Verify …")
calib_ver = score_verify(calib_rows)

# ── Sweep PLM temperature (pure numpy after this point) ──────────────────────
print("\n── PLM temperature sweep ──")
TEMPS = [0.3, 0.5, 0.7, 1.0, 1.3, 1.5]
best_plm_temp, best_plm_map = 1.0, -1.0
for T in TEMPS:
    preds = ensemble(calib_plm, calib_cot, calib_ver,
                     plm_temp=T, w_plm=1.0, w_cot=0.0, w_ver=0.0)
    m = map_at_3(preds, calib_labels)
    print(f"  T={T:.1f}  MAP@3={m:.5f}")
    if m > best_plm_map:
        best_plm_map, best_plm_temp = m, T
print(f"  → best PLM temp = {best_plm_temp}  MAP@3={best_plm_map:.5f}")

# ── Individual method scores at best PLM temp ─────────────────────────────────
map_plm_only = map_at_3(
    ensemble(calib_plm, calib_cot, calib_ver,
             plm_temp=best_plm_temp, w_plm=1.0, w_cot=0.0, w_ver=0.0),
    calib_labels)
map_cot_only = map_at_3(
    ensemble(calib_plm, calib_cot, calib_ver,
             plm_temp=best_plm_temp, w_plm=0.0, w_cot=1.0, w_ver=0.0),
    calib_labels)
map_ver_only = map_at_3(
    ensemble(calib_plm, calib_cot, calib_ver,
             plm_temp=best_plm_temp, w_plm=0.0, w_cot=0.0, w_ver=1.0),
    calib_labels)
print(f"\nIndividual MAP@3:  PLM={map_plm_only:.5f}  CoT={map_cot_only:.5f}  Ver={map_ver_only:.5f}")

# ── Grid search for best weights ─────────────────────────────────────────────
print("\n── Ensemble weight grid search ──")
best_combo = (1.0, 0.0, 0.0)
best_ens_map = -1.0
for wp in [0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]:
    for wc in [0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]:
        wv = round(1.0 - wp - wc, 2)
        if wv < 0: continue
        preds = ensemble(calib_plm, calib_cot, calib_ver,
                         plm_temp=best_plm_temp, w_plm=wp, w_cot=wc, w_ver=wv)
        m = map_at_3(preds, calib_labels)
        if m > best_ens_map:
            best_ens_map = m
            best_combo   = (wp, wc, wv)

print(f"Best weights  PLM={best_combo[0]}  CoT={best_combo[1]}  Ver={best_combo[2]}")
print(f"Best ensemble MAP@3 = {best_ens_map:.5f}")


Running 3 scoring methods on 200 calib rows …
[1/3] PLM …


PLM perms:   0%|          | 0/5 [00:00<?, ?it/s]

[2/3] CoT …


CoT:   0%|          | 0/50 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[3/3] Verify …


Verify:   0%|          | 0/200 [00:00<?, ?it/s]


── PLM temperature sweep ──
  T=0.3  MAP@3=0.97250
  T=0.5  MAP@3=0.97250
  T=0.7  MAP@3=0.97250
  T=1.0  MAP@3=0.97250
  T=1.3  MAP@3=0.97250
  T=1.5  MAP@3=0.97250
  → best PLM temp = 0.3  MAP@3=0.97250

Individual MAP@3:  PLM=0.97250  CoT=0.87667  Ver=0.86500

── Ensemble weight grid search ──
Best weights  PLM=0.3  CoT=0.3  Ver=0.4
Best ensemble MAP@3 = 0.97417


In [11]:
print("Running all 3 methods on full test set …")
test_rows = [row for _,row in test.iterrows()]

print("[1/3] PLM …")
test_plm = score_plm(test_rows, batch_size=8)

print("[2/3] CoT …")
test_cot = score_cot(test_rows, batch_size=4)

print("[3/3] Verify …")
test_ver = score_verify(test_rows)

predictions = ensemble(
    test_plm, test_cot, test_ver,
    plm_temp=best_plm_temp,
    w_plm=best_combo[0], w_cot=best_combo[1], w_ver=best_combo[2],
)

submission = pd.DataFrame({"ID": test["id"], "Prediction": predictions})
submission.to_csv("submission_cl.csv", index=False)
print("Saved submission.csv")
submission.head(10)


Running all 3 methods on full test set …
[1/3] PLM …


PLM perms:   0%|          | 0/5 [00:00<?, ?it/s]

[2/3] CoT …


CoT:   0%|          | 0/125 [00:00<?, ?it/s]

[3/3] Verify …


Verify:   0%|          | 0/500 [00:00<?, ?it/s]

Saved submission.csv


,ID,Prediction
0,1,A E B
1,2,B A D
2,3,B C E
3,4,E D C
4,5,C E D
5,6,D C A
6,7,E A C
7,8,B E A
8,9,C D E
9,10,B D E


In [ ]:
top1 = [p.split()[0] for p in predictions]
print("Top-1 distribution:", dict(sorted(Counter(top1).items())))
assert len(submission) == len(test)
assert submission['Prediction'].str.split().str.len().eq(3).all()
print("All checks passed.")
